# L5: Multi-agent Collaboration for Financial Analysis

In this lesson, you will learn ways for making agents collaborate with each other.

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, APIs and LLM

In [2]:
from crewai import Agent, Task, Crew, Process

In [3]:
import os
from utils import get_openai_api_key, get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

## crewAI Tools

In [4]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

## Creating Agents

In [5]:
data_analyst_agent = Agent(
    role="Data Analyst",
    goal="Monitor and analyze market data in real-time "
         "to identify trends and predict market movements.",
    backstory="Specializing in financial markets, this agent "
              "uses statistical modeling and machine learning "
              "to provide crucial insights. With a knack for data, "
              "the Data Analyst Agent is the cornerstone for "
              "informing trading decisions.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [6]:
trading_strategy_agent = Agent(
    role="Trading Strategy Developer",
    goal="Develop and test various trading strategies based "
         "on insights from the Data Analyst Agent.",
    backstory="Equipped with a deep understanding of financial "
              "markets and quantitative analysis, this agent "
              "devises and refines trading strategies. It evaluates "
              "the performance of different approaches to determine "
              "the most profitable and risk-averse options.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [7]:
execution_agent = Agent(
    role="Trade Advisor",
    goal="Suggest optimal trade execution strategies "
         "based on approved trading strategies.",
    backstory="This agent specializes in analyzing the timing, price, "
              "and logistical details of potential trades. By evaluating "
              "these factors, it provides well-founded suggestions for "
              "when and how trades should be executed to maximize "
              "efficiency and adherence to strategy.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [8]:
risk_management_agent = Agent(
    role="Risk Advisor",
    goal="Evaluate and provide insights on the risks "
         "associated with potential trading activities.",
    backstory="Armed with a deep understanding of risk assessment models "
              "and market dynamics, this agent scrutinizes the potential "
              "risks of proposed trades. It offers a detailed analysis of "
              "risk exposure and suggests safeguards to ensure that "
              "trading activities align with the firm’s risk tolerance.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

## Creating Tasks

In [9]:
# Task for Data Analyst Agent: Analyze Market Data
data_analysis_task = Task(
    description=(
        "Continuously monitor and analyze market data for "
        "the selected stock ({stock_selection}). "
        "Use statistical modeling and machine learning to "
        "identify trends and predict market movements."
    ),
    expected_output=(
        "Insights and alerts about significant market "
        "opportunities or threats for {stock_selection}."
    ),
    agent=data_analyst_agent,
)

In [10]:
# Task for Trading Strategy Agent: Develop Trading Strategies
strategy_development_task = Task(
    description=(
        "Develop and refine trading strategies based on "
        "the insights from the Data Analyst and "
        "user-defined risk tolerance ({risk_tolerance}). "
        "Consider trading preferences ({trading_strategy_preference})."
    ),
    expected_output=(
        "A set of potential trading strategies for {stock_selection} "
        "that align with the user's risk tolerance."
    ),
    agent=trading_strategy_agent,
)


In [11]:
# Task for Trade Advisor Agent: Plan Trade Execution
execution_planning_task = Task(
    description=(
        "Analyze approved trading strategies to determine the "
        "best execution methods for {stock_selection}, "
        "considering current market conditions and optimal pricing."
    ),
    expected_output=(
        "Detailed execution plans suggesting how and when to "
        "execute trades for {stock_selection}."
    ),
    agent=execution_agent,
)


In [12]:
# Task for Risk Advisor Agent: Assess Trading Risks
risk_assessment_task = Task(
    description=(
        "Evaluate the risks associated with the proposed trading "
        "strategies and execution plans for {stock_selection}. "
        "Provide a detailed analysis of potential risks "
        "and suggest mitigation strategies."
    ),
    expected_output=(
        "A comprehensive risk analysis report detailing potential "
        "risks and mitigation recommendations for {stock_selection}."
    ),
    agent=risk_management_agent,
)

## Creating the Crew
- The `Process` class helps to delegate the workflow to the Agents (kind of like a Manager at work)
- In the example below, it will run this hierarchically.
- `manager_llm` lets you choose the "manager" LLM you want to use.

In [13]:
from langchain_openai import ChatOpenAI

# Define the crew with agents and tasks
financial_trading_crew = Crew(
    agents=[data_analyst_agent, 
            trading_strategy_agent, 
            execution_agent, 
            risk_management_agent],
    
    tasks=[data_analysis_task, 
           strategy_development_task, 
           execution_planning_task, 
           risk_assessment_task],
    
    manager_llm=ChatOpenAI(model="gpt-4o-mini", 
                           temperature=0.7),
    process=Process.hierarchical,
    verbose=True
)

## Running the Crew

- Set the inputs for the execution of the crew.

In [14]:
# Example data for kicking off the process
financial_trading_inputs = {
    'stock_selection': 'AAPL',
    'initial_capital': '100000',
    'risk_tolerance': 'Medium',
    'trading_strategy_preference': 'Day Trading',
    'news_impact_consideration': True
}

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [15]:
### this execution will take some time to run
result = financial_trading_crew.kickoff(inputs=financial_trading_inputs)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 6a018d18-9eea-428a-8bef-ebf83378eeca                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: Continuously monitor and analyze market data for the selected stock (AAPL). Use statistical modeling     │
│  and machine learning to identify trends and predict market movements.                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

Output()

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Thought: I need to gather relevant market data for the selected stock (AAPL) in order to analyze trends and    │
│  predict market movements. This requires me to find a reliable source for continuous market data and insights.  │
│                                                                                                                 │
│  Using Tool: Search the internet with Serper                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Risk Analysis Report for AAPL Trading Strategies**                                                           │
│                                                                                                                 │
│  **1. Momentum Trading Strategy:**                                                                              │
│  - **Potential Risks:**                                                                                         │
│    - **High Volatility:** The reliance on market momentum can lead to significant losses if prices fluctuate    │
│  unexpectedly within a short period.                                                                            │
│    - **Trading Costs:** Frequent buying and selling may incur high transaction costs, eroding overall           │
│  profitability.                                                                                                 │
│    - **Market Dependence:** Momentum trading heavily depends on clear market trends. In periods of low          │
│  volatility or consolidation, opportunities may be scarce.                                                      │
│    - **Short-term Focus:** This strategy's emphasis on immediate results can cause traders to overlook          │
│  long-term investment benefits.                                                                                 │
│                                                                                                                 │
│  - **Mitigation Strategies:**                                                                                   │
│    - Implement stop-loss orders to cap potential losses.                                                        │
│    - Conduct thorough market research to make informed trading decisions.                                       │
│    - Define clear entry and exit points to manage trades effectively.                                           │
│    - Closely monitor market trends to adjust strategies as needed.                                              │
│                                                                                                                 │
│  **2. Moving Average Crossover:**                                                                               │
│  - **Potential Risks:**                                                                                         │
│    - **Lagging Indicators:** Moving averages are based on historical data, which may not predict future price   │
│  movements accurately.                                                                                          │
│    - **False Signals:** Market whipsaws can trigger buy/sell signals that result in losses.                     │
│                                                                                                                 │
│  - **Mitigation Strategies:**                                                                                   │
│    - Combine moving averages with additional indicators (e.g., RSI or MACD) to confirm signals.                 │
│    - Use shorter-term moving averages for faster signals during volatile periods.                               │
│                                                                                                                 │
│  **3. Breakout Trading:**                              

Output()

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 4010c7fd-9aeb-41c7-b63e-348b5a357883                                                                     │
│  Agent: Crew Manager                                                                                            │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 6a018d18-9eea-428a-8bef-ebf83378eeca                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: **Risk Analysis Report for AAPL Trading Strategies**                                             │
│                                                                                                                 │
│  **1. Momentum Trading Strategy:**                                                                              │
│  - **Potential Risks:**                                                                                         │
│    - **High Volatility:** The reliance on market momentum can lead to significant losses if prices fluctuate    │
│  unexpectedly within a short period.                                                                            │
│    - **Trading Costs:** Frequent buying and selling may incur high transaction costs, eroding overall           │
│  profitability.                                                                                                 │
│    - **Market Dependence:** Momentum trading heavily depends on clear market trends. In periods of low          │
│  volatility or consolidation, opportunities may be scarce.                                                      │
│    - **Short-term Focus:** This strategy's emphasis on immediate results can cause traders to overlook          │
│  long-term investment benefits.                                                                                 │
│                                                                                                                 │
│  - **Mitigation Strategies:**                                                                                   │
│    - Implement stop-loss orders to cap potential losses.                                                        │
│    - Conduct thorough market research to make informed trading decisions.                                       │
│    - Define clear entry and exit points to manage trades effectively.                                           │
│    - Closely monitor market trends to adjust strategies as needed.                                              │
│                                                                                                                 │
│  **2. Moving Average Crossover:**                                                                               │
│  - **Potential Risks:**                                                                                         │
│    - **Lagging Indicators:** Moving averages are based on historical data, which may not predict future price   │
│  movements accurately.                                                                                          │
│    - **False Signals:** Market whipsaws can trigger buy/sell signals that result in losses.                     │
│                                                                                                                 │
│  - **Mitigation Strategies:**                                                                                   │
│    - Combine moving averages with additional indicators (e.g., RSI or MACD) to confirm signals.                 │
│    - Use shorter-term moving averages for faster signals during volatile periods.                               │
│                                                       

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

- Display the final result as Markdown.

In [16]:
from IPython.display import Markdown
Markdown(result.raw)

**Risk Analysis Report for AAPL Trading Strategies**

**1. Momentum Trading Strategy:**
- **Potential Risks:**
  - **High Volatility:** The reliance on market momentum can lead to significant losses if prices fluctuate unexpectedly within a short period.
  - **Trading Costs:** Frequent buying and selling may incur high transaction costs, eroding overall profitability.
  - **Market Dependence:** Momentum trading heavily depends on clear market trends. In periods of low volatility or consolidation, opportunities may be scarce.
  - **Short-term Focus:** This strategy's emphasis on immediate results can cause traders to overlook long-term investment benefits.
  
- **Mitigation Strategies:**
  - Implement stop-loss orders to cap potential losses.
  - Conduct thorough market research to make informed trading decisions.
  - Define clear entry and exit points to manage trades effectively.
  - Closely monitor market trends to adjust strategies as needed.
  
**2. Moving Average Crossover:**
- **Potential Risks:**
  - **Lagging Indicators:** Moving averages are based on historical data, which may not predict future price movements accurately.
  - **False Signals:** Market whipsaws can trigger buy/sell signals that result in losses.
  
- **Mitigation Strategies:**
  - Combine moving averages with additional indicators (e.g., RSI or MACD) to confirm signals.
  - Use shorter-term moving averages for faster signals during volatile periods.

**3. Breakout Trading:**
- **Potential Risks:**
  - **False Breakouts:** Price may break support or resistance levels only to revert back, leading to losses.
  - **Volume Misinterpretation:** Low volume during a breakout can signal a lack of conviction in the move.
  
- **Mitigation Strategies:**
  - Wait for confirmation of breakouts (e.g., close above resistance with high volume).
  - Use alerts from the Data Analyst to monitor key levels and market sentiment.

**4. Scalping:**
- **Potential Risks:**
  - **High Transaction Costs:** The strategy's nature can lead to significant transaction fees that cut into profits.
  - **Market Noise:** Rapid price fluctuations can result in losses rather than gains if trades are not executed promptly.
  
- **Mitigation Strategies:**
  - Set strict profit targets and loss limits for each trade to minimize risks.
  - Focus on liquid stocks with lower spreads to reduce transaction costs.

**5. News-Based Trading:**
- **Potential Risks:**
  - **Market Reaction Delays:** Immediate reactions to news may not reflect the actual impact on AAPL stock, leading to poor trading decisions.
  - **Overreaction to News:** Traders may react impulsively to news, resulting in losses.
  
- **Mitigation Strategies:**
  - Utilize sentiment analysis tools to gauge market reaction accurately.
  - Monitor for secondary news that may influence initial market responses.

**6. Risk Management:**
- **Potential Risks:**
  - **Inadequate Risk Management:** Failure to set effective stop-loss and take-profit orders can expose traders to significant losses.
  
- **Mitigation Strategies:**
  - Establish clear stop-loss orders at 2% below the purchase price and take-profit orders at 3% above the purchase price.
  - Regularly review and adjust risk management strategies based on market conditions.

By addressing these potential risks and implementing the recommended mitigation strategies, the team can enhance their trading effectiveness and protect against significant losses while executing the proposed trading strategies for AAPL.